# CS 229 - HW1 EC1: Robustness & Variance

Run the same prompts multiple times with `do_sample=True` and `temperature > 0`. Measure the variance in answers and consistency across the different methods.


## Setup


In [1]:
!pip install -q transformers torch peft accelerate tqdm

In [2]:
import torch, re, numpy as np, matplotlib.pyplot as plt
from tqdm import tqdm
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
LORA_PATH = "drive/MyDrive/lora_alien_calcgpt"  # adjust path as needed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
print(f"Loaded {MODEL_NAME}")

# Load LoRA fine-tuned model
lora_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
lora_model = PeftModel.from_pretrained(lora_model, LORA_PATH)
lora_model.eval()
print(f"Loaded LoRA adapter from {LORA_PATH}")


Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-0.5B-Instruct


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

ValueError: Can't find 'adapter_config.json' at 'drive/MyDrive/lora_alien_calcgpt'

## Load Data


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
data = torch.load('drive/MyDrive/HW 1 Alien CalcGPT.pt', weights_only=False)


In [ ]:
# data = torch.load('hw1_data.pt', weights_only=False)
train_problems  = data['train_problems']
train_answers   = data['train_answers']
train_levels    = data['train_levels']
train_standard  = data['train_standard']
test_problems   = data['test_problems']
test_answers    = data['test_answers']
test_levels     = data['test_levels']
test_standard   = data['test_standard']
operators       = data['operators']
print(f"Train: {len(train_problems)}, Test: {len(test_problems)}")


Train: 300, Test: 150


## Helpers


In [ ]:
def generate_response(messages, model, tokenizer, max_new_tokens=64, temperature=0.0, do_sample=False):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=do_sample, temperature=temperature if do_sample else None,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

def extract_answer(response):
    match = re.search(r"Final Answer:\s*(-?\d+)", response)
    return int(match.group(1)) if match else None


## Prompt Builders (from HW1)


In [ ]:
def build_base_prompt(problem):
    return [{'role': "user", 'content': f"What is {problem}? Respond with: Final Answer: <number>"}]

def build_system_prompt(problem):
    sys = "You are an expert mathematician who understands the following alien operators:\n"
    for name, desc in operators.items():
        sys += f"- {name}(...) = {desc}\n"
    return [{'role': "system", 'content': sys}, {'role': "user", 'content': f"What is {problem}? Respond with: Final Answer: <number>"}]

def build_cot_prompt(problem):
    sys = "You are an expert mathematician who understands the following alien operators:\n"
    for name, desc in operators.items():
        sys += f"- {name}(...) = {desc}\n"
    sys += "Explain your reasoning step-by-step. You MUST conclude with: 'Final Answer: <number>'.\n"
    return [{'role': "system", 'content': sys}, {'role': "user", 'content': f"What is {problem}? Show your work. End with: Final Answer: <number>"}]

# --- ICL ---
def get_icl_examples(train_problems, train_answers, train_levels, n_per_level=1):
    res = []
    for difficulty in [1, 2, 3]:
        idxs = [i for i in range(len(train_problems)) if train_levels[i].item() == difficulty]
        for i in idxs[:n_per_level]:
            res.append((train_problems[i], train_answers[i].item()))
    return res

icl_examples = get_icl_examples(train_problems, train_answers, train_levels, n_per_level=1)

def build_icl_prompt(problem):
    sys = "You are an expert mathematician who understands the following alien operators:\n"
    for name, desc in operators.items():
        sys += f"- {name}(...) = {desc}\n"
    messages = [{"role": "system", "content": sys}]
    for ex_prob, ex_ans in icl_examples:
        messages.append({"role": "user", "content": f"What is {ex_prob}? Respond with: Final Answer: <number>"})
        messages.append({"role": "assistant", "content": f"Final Answer: {ex_ans}"})
    messages.append({"role": "user", "content": f"What is {problem}? Respond with: Final Answer: <number>"})
    return messages

# (prompt_fn, max_tokens, model_to_use)
methods = {
    "Base": (build_base_prompt, 64, model),
    "System": (build_system_prompt, 64, model),
    "CoT": (build_cot_prompt, 512, model),
    "ICL": (build_icl_prompt, 64, model),
    "LoRA": (build_system_prompt, 64, lora_model),
}


## Variance Experiment

For each method, run every test problem `N_RUNS` times with `temperature > 0` and measure how often the answer changes.


In [ ]:
N_RUNS = 5
TEMPERATURE = 0.7
variance_results = {}

for method_name, (prompt_fn, max_tokens, method_model) in methods.items():
    print(f"\n{'='*50}")
    print(f"Method: {method_name} | {N_RUNS} runs per problem | temp={TEMPERATURE}")
    print(f"{'='*50}")
    
    all_run_preds = []  # shape: (N_RUNS, n_problems)
    for run in range(N_RUNS):
        preds = []
        for problem in tqdm(test_problems, desc=f"{method_name} run {run+1}"):
            resp = generate_response(prompt_fn(problem), method_model, tokenizer,
                                     max_new_tokens=max_tokens, temperature=TEMPERATURE, do_sample=True)
            preds.append(extract_answer(resp))
        all_run_preds.append(preds)
    
    # Compute per-problem consistency
    n_problems = len(test_problems)
    consistency = []  # fraction of runs that agree with the mode
    mode_correct = []  # whether the mode answer is correct
    unique_counts = []  # number of unique answers per problem
    
    for j in range(n_problems):
        answers_j = [all_run_preds[r][j] for r in range(N_RUNS)]
        counter = Counter(answers_j)
        mode_ans, mode_count = counter.most_common(1)[0]
        consistency.append(mode_count / N_RUNS)
        mode_correct.append(mode_ans == test_answers[j].item())
        unique_counts.append(len(counter))
    
    avg_consistency = np.mean(consistency)
    mode_accuracy = np.mean(mode_correct)
    avg_unique = np.mean(unique_counts)
    
    variance_results[method_name] = {
        "all_run_preds": all_run_preds,
        "consistency": consistency,
        "mode_correct": mode_correct,
        "unique_counts": unique_counts,
        "avg_consistency": avg_consistency,
        "mode_accuracy": mode_accuracy,
        "avg_unique": avg_unique,
    }
    
    print(f"  Avg consistency: {avg_consistency:.3f}")
    print(f"  Mode accuracy:   {mode_accuracy:.3f}")
    print(f"  Avg unique answers per problem: {avg_unique:.2f}")



Method: Base | 5 runs per problem | temp=0.7


Base run 1:  79%|███████▉  | 119/150 [00:30<00:11,  2.64it/s]

## Results


In [ ]:
# Bar chart: consistency and mode accuracy by method
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

names = list(variance_results.keys())
consistencies = [variance_results[m]["avg_consistency"] * 100 for m in names]
accuracies = [variance_results[m]["mode_accuracy"] * 100 for m in names]
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12']

axes[0].bar(names, consistencies, color=colors)
axes[0].set_ylabel("Consistency (%)")
axes[0].set_title("Average Answer Consistency (across runs)")
axes[0].set_ylim(0, 100)
for b, v in zip(axes[0].patches, consistencies):
    axes[0].text(b.get_x() + b.get_width()/2, b.get_height()+1, f"{v:.1f}%", ha='center')

axes[1].bar(names, accuracies, color=colors)
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Mode-Answer Accuracy")
axes[1].set_ylim(0, 100)
for b, v in zip(axes[1].patches, accuracies):
    axes[1].text(b.get_x() + b.get_width()/2, b.get_height()+1, f"{v:.1f}%", ha='center')

plt.tight_layout()
plt.show()


In [ ]:
# Histogram of unique answers per problem
fig, axes = plt.subplots(1, len(names), figsize=(4*len(names), 4), sharey=True)
for ax, m, c in zip(axes, names, colors):
    ax.hist(variance_results[m]["unique_counts"], bins=range(1, N_RUNS+2),
            align='left', color=c, edgecolor='black', alpha=0.8)
    ax.set_xlabel("# Unique Answers")
    ax.set_title(m)
    ax.set_xticks(range(1, N_RUNS+1))
axes[0].set_ylabel("# Problems")
plt.suptitle(f"Distribution of Unique Answers per Problem (temp={TEMPERATURE}, {N_RUNS} runs)")
plt.tight_layout()
plt.show()


In [ ]:
# Per-level consistency breakdown
for m in names:
    print(f"\n--- {m} ---")
    for level in [1, 2, 3]:
        idxs = [i for i in range(len(test_levels)) if test_levels[i].item() == level]
        avg_c = np.mean([variance_results[m]["consistency"][i] for i in idxs])
        avg_u = np.mean([variance_results[m]["unique_counts"][i] for i in idxs])
        print(f"  Level {level}: consistency={avg_c:.3f}, avg unique={avg_u:.2f}")


## Analysis

*TODO: Write a short paragraph (4–6 sentences). Which methods are most/least reliable? Does consistency correlate with accuracy? How does difficulty level affect variance?*
